# Analytic Solov'ev equilibrium

Construct a constant-source Grad–Shafranov solution from physical boundary constraints, compare analytic and discretized fields, and export it to VAFT's lightweight in-memory representation.

In [ ]:
import numpy as np
from vaft.data.equilibrium import SolovevConstraint
from vaft.process.equilibrium import evaluate_solovev, solve_solovev_constraints, solovev_to_equilibrium

constraints = [
    SolovevConstraint(1.0, 0.0, "psi", -1.0),
    SolovevConstraint(0.7, 0.0, "psi", 0.0),
    SolovevConstraint(1.3, 0.0, "psi", 0.0),
    SolovevConstraint(1.0, 0.4, "psi", 0.0),
    SolovevConstraint(1.0, -0.4, "psi", 0.0),
    SolovevConstraint(1.0, 0.0, "dpsi_dr", 0.0),
]
model = solve_solovev_constraints(constraints, pprime=-1e5, ffprime=0.0, rref=1.0)
print("rank/residual:", model.rank, model.residual_norm)
r = np.linspace(0.5, 1.5, 151); z = np.linspace(-0.7, 0.7, 151)
rm, zm = np.meshgrid(r, z, indexing="ij")
analytic = evaluate_solovev(model, rm, zm)
equilibrium = solovev_to_equilibrium(model, r, z, magnetic_axis=(1.0, 0.0), convention=11)
print("portable grid:", equilibrium.psi.shape, "LCFS points:", equilibrium.lcfs.r.size)

In [ ]:
dpsi_dr = np.gradient(equilibrium.psi, r, axis=0, edge_order=2)
dpsi_dz = np.gradient(equilibrium.psi, z, axis=1, edge_order=2)
br_grid = -dpsi_dz/rm; bz_grid = dpsi_dr/rm
np.testing.assert_allclose(br_grid[3:-3, 3:-3], analytic["b_r"][3:-3, 3:-3], rtol=1e-2, atol=3e-2)
np.testing.assert_allclose(bz_grid[3:-3, 3:-3], analytic["b_z"][3:-3, 3:-3], rtol=1e-2, atol=3e-2)
print("Analytic and discretized poloidal fields agree within the grid tolerance.")